# Machine Learning CIE 2 — Dataset Preparation & Exploratory Data Analysis

**Activity 1:** Dataset Preparation Activity  
**Unit I:** Foundations & Data Preparation  
**Student:** Harshavardhan Emade  
**Roll Number:** AD3149  
**Branch:** AI & DS  
**Target level:** Second Year Engineering — Machine Learning Laboratory

## 1. Project objective
Build a reproducible, beginner-friendly data preparation pipeline using Python and the public Titanic passenger dataset. The workflow covers dataset ingestion and inspection, missing-value treatment, duplicate removal, categorical consistency, exploratory data analysis (EDA), one-hot encoding, feature scaling, validation and CSV export.

**Scope boundary:** This activity is limited to foundational data preparation. No machine-learning prediction model is trained or evaluated.

## 2. Dataset and public source
- Dataset: Titanic Passenger Survival Dataset
- Repository: https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv
- Direct CSV: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv
- Format: CSV

The commonly distributed source contains 891 rows and 15 columns. The notebook checks the actual downloaded shape rather than assuming it. See the data dictionary in the README for column meanings.


## 3. Technology stack
Python 3; pandas and NumPy for tabular processing; Matplotlib and Seaborn for visualisation; scikit-learn for StandardScaler; Jupyter for execution.

In [ ]:
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from IPython.display import display

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
DATA_DIR = Path("data")
FIG_DIR = Path("reports/figures")
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = DATA_DIR / "titanic.csv"

if not RAW_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, RAW_PATH)

df_raw = pd.read_csv(RAW_PATH)
print("Raw shape:", df_raw.shape)
display(df_raw.head())


## 4. Initial data audit
Inspect data types, descriptive statistics, null counts, and exact duplicate counts. These values are calculated from the downloaded file at runtime.

In [ ]:
df_raw.info()
display(df_raw.describe(include="all").T)
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2),
    "unique_values": df_raw.nunique(dropna=True)
})
display(audit)
print("Exact duplicate rows in raw data:", df_raw.duplicated().sum())


## 5. Data cleaning and preparation

### 5.1 Missing-value treatment
- `deck`: drop because it is highly sparse; imputing most of this column would introduce substantial synthetic information.
- `age`: median imputation, which is less sensitive to extreme values than the mean.
- `embarked`: mode imputation because it is categorical.
- Any additional missing values are audited after cleaning; the notebook raises an error if unexpected nulls remain.

### 5.2 Duplicates and consistency
The notebook removes exact duplicates after dropping the sparse `deck` field. This makes the duplicate definition explicit: rows identical across the retained fields are treated as duplicates. It trims whitespace in text columns and removes redundant columns (`alive`, `class`, `embark_town`) because they duplicate `survived`, `pclass`, and `embarked`, respectively.

**Data-leakage note:** In a predictive workflow, imputation and scaling parameters should be fitted on the training split only and then applied to validation/test data. This teaching activity prepares the full dataset for demonstration and does not claim to be a leakage-safe model training pipeline.


In [ ]:
df = df_raw.copy()

# Drop the very sparse cabin-deck field before checking duplicate rows.
if "deck" in df.columns:
    df = df.drop(columns=["deck"])

# Impute missing values using column-appropriate statistics.
if "age" in df.columns:
    df["age"] = df["age"].fillna(df["age"].median())
if "embarked" in df.columns and df["embarked"].isna().any():
    df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

before = len(df)
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Rows before deduplication:", before)
print("Duplicate rows removed:", duplicate_count)

# Normalize surrounding whitespace in categorical text.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Remove columns that duplicate information represented by other fields.
redundant_cols = [c for c in ["alive", "class", "embark_town"] if c in df.columns]
df = df.drop(columns=redundant_cols)

print("Shape after cleaning:", df.shape)
display(df.isna().sum().to_frame("missing_after_cleaning"))


## 6. Exploratory Data Analysis
Generate three plots and save them at 300 DPI. Interpretations should be based on the plots produced by the actual run; avoid treating association as causation.

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 5))
sns.histplot(df["age"], bins=25, kde=True)
plt.axvline(df["age"].median(), linestyle="--", label=f"Median: {df['age'].median():.1f}")
plt.title("Distribution of Passenger Ages")
plt.xlabel("Age (years)")
plt.ylabel("Passenger count")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "age_distribution.png", dpi=300)
plt.show()

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="survived")
ax.set_xticks([0, 1], ["Did not survive (0)", "Survived (1)"])
plt.title("Passenger Survival Counts")
plt.xlabel("Survival status")
plt.ylabel("Passenger count")
plt.tight_layout()
plt.savefig(FIG_DIR / "survival_counts.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="fare", hue="survived", alpha=0.65)
plt.title("Passenger Age vs Ticket Fare")
plt.xlabel("Age (years)")
plt.ylabel("Fare")
plt.tight_layout()
plt.savefig(FIG_DIR / "age_vs_fare.png", dpi=300)
plt.show()

survival_summary = df["survived"].value_counts().sort_index().rename(index={0:"Did not survive",1:"Survived"})
display(survival_summary.to_frame("passenger_count"))
print("Median age:", round(df["age"].median(), 2))


## 7. Categorical encoding and numerical scaling

- Boolean features such as `adult_male` and `alone` are converted to integer flags (0/1).
- `pclass` is retained as an ordinal integer.
- Nominal columns `sex`, `embarked`, and `who` are one-hot encoded using `drop_first=True` to omit one reference category per variable.
- Continuous columns `age` and `fare` are standardized using `StandardScaler`.

Standardization uses \(z = (x - \mu) / \sigma\). Binary flags, family counts, ordinal class, and the target `survived` are not scaled.


In [ ]:
for col in ["adult_male", "alone"]:
    if col in df.columns:
        df[col] = df[col].astype(int)

nominal_cols = [c for c in ["sex", "embarked", "who"] if c in df.columns]
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True, dtype=int)

scale_cols = [c for c in ["age", "fare"] if c in df_encoded.columns]
scaler = StandardScaler()
df_prepared = df_encoded.copy()
df_prepared[scale_cols] = scaler.fit_transform(df_prepared[scale_cols])

display(df_prepared.head())
print("Prepared shape:", df_prepared.shape)
display(pd.DataFrame({
    "mean": df_prepared[scale_cols].mean(),
    "std_population": df_prepared[scale_cols].std(ddof=0)
}).round(6))


## 8. Validation and export
Programmatic checks confirm that no missing values or duplicate rows remain and that the target column is preserved. The notebook exports the prepared table to `prepared_titanic.csv`. Final dimensions are printed from the executed data rather than hard-coded.


In [ ]:
assert df_prepared.isna().sum().sum() == 0, "Missing values remain"
assert df_prepared.duplicated().sum() == 0, "Duplicate rows remain"
assert "survived" in df_prepared.columns, "Target column is missing"
assert set(df_prepared["survived"].dropna().unique()).issubset({0, 1})

OUTPUT_PATH = Path("prepared_titanic.csv")
df_prepared.to_csv(OUTPUT_PATH, index=False)

print("Raw dimensions:", df_raw.shape)
print("Prepared dimensions:", df_prepared.shape)
print("Missing cells after preparation:", int(df_prepared.isna().sum().sum()))
print("Duplicate rows after preparation:", int(df_prepared.duplicated().sum()))
print("Duplicate rows removed:", duplicate_count)
print("Saved:", OUTPUT_PATH.resolve())
print("All validation checks passed.")


## 9. Conclusion
This activity demonstrated a reproducible data preparation lifecycle: ingestion, inspection, missing-value remediation, duplicate removal, categorical cleanup, EDA, encoding, scaling, validation and export. The prepared CSV is suitable as a starting point for a subsequent modelling exercise, with training/test split and leakage-safe preprocessing to be implemented at that later stage.

## 10. Academic acknowledgement
This activity was completed as part of the Machine Learning CIE 2 laboratory work. Faculty reference and learning resources: https://github.com/Princy2129

**No prediction model was trained or evaluated in this activity.**
